In [11]:
import pandas as pd
import numpy as np
import time
import os
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"
config

{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg',
  'file3': '../data/raw/fichier_diffusion_2024.xlsx',
  'file4': '../data/raw/2024-10-01_performance_fixed_tiles.parquet'},
 'output_data': {'file1': '../data/clean/perf_admin_fr_df.csv',
  'file2': '../data/clean/file2_cleaned.csv'}}

In [5]:
df = pd.read_csv(config['output_data']['file1'], dtype={
        'code_insee_dept': str,
        'code_insee_comm': str,
        'zip_code': str
    })
df.head()

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,department_name,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural


In [8]:
# Create latency categories
def latency_cat(df):
    df1 = df.copy()
    bins = [0, 20, 50, 100, 150, float('inf')]
    labels = ["Excellent", "Good", "Acceptable", "Fair", "Poor"]
    df1['load_lat_ms'] = (df1['avg_lat_down_ms'] + df1['avg_lat_up_ms'])/2
    df1["latency_category"] = pd.cut(
                                        df1['load_lat_ms'],
                                        bins = bins,
                                        labels = labels,
                                        right = False
                                      )
  
    return df1

df1 = latency_cat(df)
df1.head()

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,...,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS,load_lat_ms,latency_category
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural,779.5,Poor
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural,129.5,Fair
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural,172.5,Poor
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural,408.5,Poor
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural,241.0,Poor


In [16]:
#Broaddband quality score
def broadband_qty_score(df):
    df1 = latency_cat(df)
    #Normalize the load_lat_ms, avg_up_mbps and 	avg_down_mbps
    #create an instance oft he normalizer
    normalizer = MinMaxScaler()

    #fit the normalizer
    df1['down_speed_norm'] = normalizer.fit_transform(df1[['avg_down_mbps']])
    df1['up_speed_norm'] = normalizer.fit_transform(df1[['avg_up_mbps']])
    df1['load_lat_norm'] = 1 - normalizer.fit_transform(df1[['load_lat_ms']])

    #Calculate the borandband score
    df1['boradband_score%'] = round((df1['down_speed_norm']*0.5 + df1['up_speed_norm']*0.2 + df1['load_lat_norm']*0.3)*100, 2)

    return df1

df1 = broadband_qty_score(df)
df1.to_csv(f"{output_dir}fixed_internet_perf_clean.csv", index=False, encoding= "utf-8", sep = ",")
df1.head()
  

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,...,comm_population,zip_code,superficie_cadastrale,LIBDENS,load_lat_ms,latency_category,down_speed_norm,up_speed_norm,load_lat_norm,boradband_score%
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,...,11484,50440,14870,Rural,779.5,Poor,0.070583,0.095568,0.911615,32.79
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,...,11484,50440,14870,Rural,129.5,Fair,0.068909,0.133680,0.985744,35.69
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,...,11484,50440,14870,Rural,172.5,Poor,0.030486,0.025032,0.980841,31.45
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,...,11484,50440,14870,Rural,408.5,Poor,0.085702,0.084172,0.953926,34.59
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,...,11484,50440,14870,Rural,241.0,Poor,0.051416,0.074392,0.973028,33.25
